# Protocolo ENMC — $K_{ref}$ do NSGA-II guiando R-NSGA-II e PI-NSGA-II

Protocolo (áudio do orientador, validado):
1. **Fase 1** — NSGA-II 21×/problema → união das frentes → não dominados (Pareto combinado) → **knee de referência $K_{ref}$**.
2. **Fase 2** — R-NSGA-II 21× com `ref_points` $= K_{ref}$ → knee de **cada** uma das 21 frentes.
3. **Fase 3** — PI-NSGA-II 21× com decisor de **distância a $K_{ref}$** (alvo fixo, como o exemplo do centro do ZDT3 no pymoo) → knee de cada frente.
4. **Ideal global** — mínimo por objetivo na união das **63 frentes**.
5. **Métrica** — 21 distâncias knee→ideal para o R e 21 para o PI → tabelas (média, DP, mín, máx), boxplot, **Wilcoxon pareado** (sementes casadas: seed $i$ do R = seed $i$ do PI).

Cada fase salva seu CSV ao terminar — nada se recomputa.

## 0. Setup

In [ ]:
!pip install -q pymoo

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import gc, csv, os
import matplotlib.pyplot as plt

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.pinsga2 import PINSGA2, AutomatedDM
from pymoo.algorithms.moo.rnsga2 import RNSGA2
from pymoo.problems import get_problem
from pymoo.optimize import minimize
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from pymoo.termination import get_termination
from pymoo.util.reference_direction import select_points_with_maximum_distance

# ---------------- parâmetros canônicos ----------------
N_RUNS    = 21     # sementes casadas entre algoritmos: seed_i(R) == seed_i(PI)
POP_SIZE  = 100
N_GEN     = 300
TAU       = 30
ETA       = 4
EPSILON_R = 0.01
SEED_BASE = 42     # sementes 42..62 (21 sementes, as MESMAS nas três fases)

PROBLEMS = {
    "Truss2D"    : get_problem("truss2d"),
    "WeldedBeam" : get_problem("welded_beam"),
    "Carside"    : get_problem("carside"),   # 3 objetivos
}

SUFIXO   = f"pop{POP_SIZE}_gen{N_GEN}_runs{N_RUNS}"
CSV_F1   = f"enmc_fase1_nsga2_{SUFIXO}.csv"
CSV_F2   = f"enmc_fase2_rnsga2_{SUFIXO}.csv"
CSV_F3   = f"enmc_fase3_pinsga2_{SUFIXO}.csv"
CSV_REF  = f"enmc_referencias_{SUFIXO}.csv"

# Patch: função de valor do PI com 3+ objetivos (necessário p/ Carside)
import math
import pymoo.util.value_functions as _mvf

def _eq_constr_poly_fixed(x):
    def _M_from_len(L):
        return int(round((-1 + math.sqrt(4 * L - 3)) / 2))
    if len(x.shape) == 1:
        M = _M_from_len(x.shape[0])
        return [-(sum(x[m*M:m*M+M]) - 1) for m in range(M)]
    else:
        M = _M_from_len(x.shape[1])
        return np.array([[-(sum(x[xi, m*M:m*M+M]) - 1) for m in range(M)]
                         for xi in range(x.shape[0])])

_mvf._eq_constr_poly = _eq_constr_poly_fixed

print("Setup ok. Problemas:")
for nome, p in PROBLEMS.items():
    print(f"  {nome:11s}: {p.n_var:2d} vars, {p.n_obj} objs, {p.n_ieq_constr} restricoes")

## 1. Funções auxiliares e persistência

In [ ]:
# Funções auxiliares (reaproveitadas dos experimentos anteriores, já validadas)

def pegar_frente_rank0(resultado):
    rank, F = resultado.pop.get("rank", "F")
    frente = F[rank == 0]
    return frente if len(frente) > 0 else F

def filtrar_nao_dominados(F_conjunto):
    nds = NonDominatedSorting()
    idx = nds.do(F_conjunto, only_non_dominated_front=True)
    return F_conjunto[idx]

def normalizar(F, F_min, F_max):
    return (np.atleast_2d(F) - F_min) / (F_max - F_min + 1e-12)

def knee_point(F):
    # Knee = maior distância à reta/hiperplano que liga os extremos (Das 1999; Branke 2004).
    # Normaliza internamente; devolve o ponto no espaço ORIGINAL.
    F = np.atleast_2d(F)
    n_pts, n_obj = F.shape
    if n_pts <= 2:
        return F[0]
    Fmin, Fmax = F.min(0), F.max(0)
    Fn = (F - Fmin) / (Fmax - Fmin + 1e-12)
    extremos = Fn[np.argmin(Fn, axis=0)]
    try:
        a, *_  = np.linalg.lstsq(extremos, np.ones(n_obj), rcond=None)
        norm_a = np.linalg.norm(a)
        if not np.isfinite(norm_a) or norm_a < 1e-12:
            raise np.linalg.LinAlgError
        dist     = np.abs(Fn @ a - 1.0) / norm_a
        knee_idx = int(np.argmax(dist))
    except np.linalg.LinAlgError:
        knee_idx = int(np.argmin(np.linalg.norm(Fn - 0.5, axis=1)))
    return F[knee_idx]

# ---- persistência ----
def salvar_frentes_csv(caminho, nome_problema, algoritmo, frentes):
    novo = not os.path.exists(caminho)
    with open(caminho, "a", newline="") as fp:
        w = csv.writer(fp)
        if novo:
            w.writerow(["problema", "algoritmo", "run", "seed", "f1", "f2", "f3"])
        n = 0
        for run, F in enumerate(frentes):
            if F is None:
                continue
            for p in np.atleast_2d(F):
                w.writerow([nome_problema, algoritmo, run, SEED_BASE + run, p[0],
                            p[1] if len(p) > 1 else "",
                            p[2] if len(p) > 2 else ""])
                n += 1
    print(f"  -> {n} soluções de {nome_problema} anexadas em {caminho}")

print("Funções auxiliares definidas")

## 2. Fase 1 — NSGA-II 21× → $P^{ref}$ → $K_{ref}$  *(salva CSV ao terminar)*

In [ ]:
# ================= FASE 1 — Referência empírica (NSGA-II, 21x) =================
# Salva o CSV da fase e as referências (K_ref + régua) IMEDIATAMENTE ao terminar.

for f in [CSV_F1, CSV_REF]:
    if os.path.exists(f): os.remove(f)   # recomeço limpo desta fase

fase1 = {}
for nome, problema in PROBLEMS.items():
    print(f"\n[FASE 1 | NSGA-II] {nome}")
    frentes = []
    for run in range(N_RUNS):
        res = minimize(problema, NSGA2(pop_size=POP_SIZE),
                       get_termination("n_gen", N_GEN),
                       seed=SEED_BASE + run, verbose=False)
        f = pegar_frente_rank0(res)
        frentes.append(f)
        print(f"  run {run+1:02d}/{N_RUNS} (seed {SEED_BASE+run}) -> {len(f)} pts")
        del res; gc.collect()

    P_ref = filtrar_nao_dominados(np.vstack(frentes))     # Pareto combinado
    K_ref = knee_point(P_ref)                             # knee de referência
    regua = (P_ref.min(axis=0), P_ref.max(axis=0))        # escala p/ o DM do PI
    fase1[nome] = {"frentes": frentes, "P_ref": P_ref, "K_ref": K_ref, "regua": regua}
    print(f"  P_ref: {len(P_ref)} pts | K_ref = {np.round(K_ref, 4)}")
    salvar_frentes_csv(CSV_F1, nome, "NSGA-II", frentes)

# referências (K_ref + régua) num CSV próprio
with open(CSV_REF, "w", newline="") as fp:
    w = csv.writer(fp)
    w.writerow(["problema", "tipo", "f1", "f2", "f3"])
    for nome in PROBLEMS:
        K = fase1[nome]["K_ref"]; mn, mx = fase1[nome]["regua"]
        for tipo, v in [("K_ref", K), ("regua_min", mn), ("regua_max", mx)]:
            v = np.atleast_1d(v)
            w.writerow([nome, tipo, v[0],
                        v[1] if len(v) > 1 else "",
                        v[2] if len(v) > 2 else ""])
print(f"\nReferências salvas em {CSV_REF}")

## 3. Decisor de alvo fixo ($K_{ref}$) + PI blindado

In [ ]:
# ============ Decisor de alvo FIXO (K_ref) + PI blindado ============
# Como o exemplo do pymoo (centro do ZDT3): prefere a solução mais próxima de um ponto
# dado — aqui, o K_ref da Fase 1. Distância medida na escala da régua da Fase 1
# (min/max de P_ref) para nenhum objetivo dominar por escala (WeldedBeam!).

class KneeRefDM(AutomatedDM):
    def __init__(self, K_ref, regua):
        super().__init__()
        self.target = np.asarray(K_ref, float)
        self.Fmin, self.Fmax = regua

    def makeDecision(self, F):
        Fn = normalizar(F,           self.Fmin, self.Fmax)
        tn = normalizar(self.target, self.Fmin, self.Fmax).ravel()
        d  = np.linalg.norm(Fn - tn, axis=1)
        if d[0] < d[1]:   return 'a'
        elif d[1] < d[0]: return 'b'
        else:             return 'c'


class SafePINSGA2(PINSGA2):
    # Mesma blindagem validada antes; diferença de protocolo: o alvo do decisor é FIXO
    # (K_ref) — removida a atualização do alvo para o knee da frente corrente.
    def _advance(self, infills=None, **kwargs):
        from pymoo.algorithms.base.genetic import GeneticAlgorithm
        import pymoo.util.value_functions as mvf

        GeneticAlgorithm._advance(self, infills=infills, **kwargs)

        try:
            rank, F = self.pop.get("rank", "F")
            self.fronts = rank
            frente = F[rank == 0]
            if frente.shape[0] == 0:
                return

            self.historical_F = (np.vstack((self.historical_F, frente))
                                 if self.historical_F is not None else frente)

            to_find = self.eta if frente.shape[0] >= self.eta else frente.shape[0]
            if self.presi_signs is None:
                self.presi_signs = np.ones(frente.shape[1])
            if to_find == 0:
                return

            eta_idx    = select_points_with_maximum_distance(frente, to_find, random_state=self.random_state)
            self.eta_F = frente[eta_idx]
            self.eta_F = self.eta_F[self.eta_F[:, 0].argsort()]
            self.eta_F = np.unique(self.eta_F, axis=0)
            self.paused_F = frente
            self.prev_pop = self.pop

            if self.n_gen % self.tau != 0:
                return
            if len(self.eta_F) < 2:
                self._reset_dm_preference()
                return

            dm_ranks = (self.automated_dm.makePairwiseDecision(self.eta_F)
                        if self.automated_dm else
                        PINSGA2._get_pairwise_ranks(self.eta_F, self.presi_signs))
            if len(set(dm_ranks)) == 0:
                self._reset_dm_preference()
                return

            eta_F = self.eta_F
            while eta_F.shape[0] > 1:
                vf_res = mvf.create_poly_vf(eta_F * -1, dm_ranks.tolist(),
                                            eps_max=self.eps_max, method=self.opt_method)
                if vf_res.fit:
                    self.vf_res       = vf_res
                    self.vf_plot_flag = True
                    self.v2           = self.vf_res.vf(eta_F[dm_ranks[1] - 1] * -1).item()
                    break
                else:
                    if eta_F.shape[0] == 2:
                        self._reset_dm_preference()
                        break
                    rt       = dm_ranks[1]
                    eta_F    = np.delete(eta_F, rt - 1, axis=0)
                    dm_ranks = np.concatenate(([dm_ranks[0]], dm_ranks[2:]))
                    dm_ranks[dm_ranks > rt] -= 1

        except Exception:
            self.n_fallbacks = getattr(self, "n_fallbacks", 0) + 1
            try:
                self._reset_dm_preference()
            except Exception:
                pass
            return

print("KneeRefDM (alvo fixo = K_ref) e SafePINSGA2 definidos")

## 4. Fase 2 — R-NSGA-II com `ref_points` = $K_{ref}$  *(salva CSV ao terminar)*

In [ ]:
# ================= FASE 2 — R-NSGA-II guiado por K_ref (21x) =================
if os.path.exists(CSV_F2): os.remove(CSV_F2)

fase2 = {}
for nome, problema in PROBLEMS.items():
    K_ref = fase1[nome]["K_ref"]
    print(f"\n[FASE 2 | R-NSGA-II] {nome}  (ref_points = K_ref = {np.round(K_ref, 4)})")
    frentes, knees = [], []
    for run in range(N_RUNS):
        alg = RNSGA2(ref_points=np.atleast_2d(K_ref), pop_size=POP_SIZE,
                     epsilon=EPSILON_R, normalization="front")
        res = minimize(problema, alg, get_termination("n_gen", N_GEN),
                       seed=SEED_BASE + run, verbose=False)
        f = pegar_frente_rank0(res)
        frentes.append(f)
        knees.append(knee_point(f))     # knee de CADA frente (protocolo)
        print(f"  run {run+1:02d}/{N_RUNS} (seed {SEED_BASE+run}) -> {len(f)} pts")
        del res, alg; gc.collect()
    fase2[nome] = {"frentes": frentes, "knees": knees}
    salvar_frentes_csv(CSV_F2, nome, "R-NSGA-II", frentes)

## 5. Fase 3 — PI-NSGA-II com DM de distância a $K_{ref}$  *(salva CSV ao terminar)*

In [ ]:
# ================= FASE 3 — PI-NSGA-II guiado por K_ref (21x) =================
if os.path.exists(CSV_F3): os.remove(CSV_F3)

fase3 = {}
fallbacks_por_problema = {}
for nome, problema in PROBLEMS.items():
    K_ref = fase1[nome]["K_ref"]
    regua = fase1[nome]["regua"]
    print(f"\n[FASE 3 | PI-NSGA-II] {nome}  (DM: distância a K_ref, alvo fixo)")
    frentes, knees, fb = [], [], []
    for run in range(N_RUNS):
        try:
            dm  = KneeRefDM(K_ref, regua)
            alg = SafePINSGA2(pop_size=POP_SIZE, tau=TAU, eta=ETA,
                              ranking_type="pairwise", automated_dm=dm, verbose=False)
            res = minimize(problema, alg, get_termination("n_gen", N_GEN),
                           seed=SEED_BASE + run, verbose=False)
            f = pegar_frente_rank0(res)
            frentes.append(f)
            knees.append(knee_point(f))
            n_fb = getattr(res.algorithm, "n_fallbacks", 0)
            fb.append(n_fb)
            print(f"  run {run+1:02d}/{N_RUNS} (seed {SEED_BASE+run}) -> {len(f)} pts | fb {n_fb}/{N_GEN}")
            del res, alg, dm
        except Exception as e:
            frentes.append(None); knees.append(None)
            print(f"  run {run+1:02d}/{N_RUNS} (seed {SEED_BASE+run}) -> ERRO ignorado: {type(e).__name__}")
        gc.collect()
    fase3[nome] = {"frentes": frentes, "knees": knees}
    fallbacks_por_problema[nome] = fb
    salvar_frentes_csv(CSV_F3, nome, "PI-NSGA-II", frentes)

## 6. Métrica — 21 distâncias knee → ideal global + Wilcoxon pareado

In [ ]:
# ========== Métrica: 21 distâncias knee -> ideal global + Wilcoxon PAREADO ==========
# Ideal global = mínimo por objetivo na união das 63 frentes (protocolo).
# Distâncias em escala normalizada (ideal/nadir da união) p/ objetivos comparáveis.
# Pareamento por RUN (semente casada): só entram no Wilcoxon os runs válidos nos DOIS.

import pandas as pd
from scipy.stats import wilcoxon
from IPython.display import display

metricas = {}
for nome in PROBLEMS:
    todas = [F for F in (fase1[nome]["frentes"] + fase2[nome]["frentes"] + fase3[nome]["frentes"])
             if F is not None]
    uniao   = np.vstack(todas)
    ideal_g = uniao.min(axis=0)
    nadir_g = uniao.max(axis=0)

    def d_ideal(k):
        if k is None: return np.nan
        kn = normalizar(k, ideal_g, nadir_g).ravel()
        return float(np.linalg.norm(kn))            # ideal normalizado = origem

    d_ref = d_ideal(fase1[nome]["K_ref"])
    d_R   = np.array([d_ideal(k) for k in fase2[nome]["knees"]])   # len 21, NaN se falhou
    d_PI  = np.array([d_ideal(k) for k in fase3[nome]["knees"]])
    metricas[nome] = {"ideal_g": ideal_g, "nadir_g": nadir_g,
                      "d_ref": d_ref, "d_R": d_R, "d_PI": d_PI}

linhas = []
for nome, m in metricas.items():
    ok = ~np.isnan(m["d_R"]) & ~np.isnan(m["d_PI"])   # pares válidos (mesma semente)
    try:
        p_w = wilcoxon(m["d_R"][ok], m["d_PI"][ok])[1]
    except Exception:
        p_w = float("nan")
    for alg, d in [("R-NSGA-II", m["d_R"]), ("PI-NSGA-II", m["d_PI"])]:
        v = d[~np.isnan(d)]
        linhas.append({"Problema": nome, "Algoritmo": alg, "n": len(v),
                       "Média": f"{v.mean():.4f}", "DP": f"{v.std():.4f}",
                       "Mín": f"{v.min():.4f}", "Máx": f"{v.max():.4f}",
                       "d(K_ref, I)": f"{m['d_ref']:.4f}",
                       "p Wilcoxon (R vs PI, pareado)": f"{p_w:.4f}"})

tab = pd.DataFrame(linhas)
display(tab)
tab.to_csv("enmc_tabela_distancias.csv", index=False)
print("Tabela salva: enmc_tabela_distancias.csv")

## 7. Boxplots

In [ ]:
# ================= Boxplots das 21 distâncias =================
fig, axes = plt.subplots(1, len(PROBLEMS), figsize=(4.2*len(PROBLEMS), 4))
axes = np.atleast_1d(axes)
for ax, (nome, m) in zip(axes, metricas.items()):
    dR  = m["d_R"][~np.isnan(m["d_R"])]
    dPI = m["d_PI"][~np.isnan(m["d_PI"])]
    ax.boxplot([dR, dPI], tick_labels=["R-NSGA-II", "PI-NSGA-II"], widths=.55)
    ax.axhline(m["d_ref"], color="crimson", ls="--", lw=1.2,
               label=f"baseline d(K_ref, I) = {m['d_ref']:.3f}")
    ax.set_title(nome); ax.grid(alpha=.3); ax.legend(fontsize=8)
axes[0].set_ylabel("distância knee → ideal global (norm.)")
fig.suptitle("Protocolo ENMC — 21 execuções por algoritmo", fontweight="bold")
fig.tight_layout(); plt.show()

## 8. Frentes com $K_{ref}$, knees e ideal

*(paleta alinhada à do notebook de análise: R-NSGA-II azul, PI-NSGA-II vermelho)*

In [ ]:
# ============ Frentes de Pareto com K_ref, os 21 knees e o ideal ============
def plot_problema(nome):
    m = metricas[nome]
    ideal_g, nadir_g = m["ideal_g"], m["nadir_g"]
    P_ref = normalizar(fase1[nome]["P_ref"], ideal_g, nadir_g)
    n_obj = P_ref.shape[1]

    # Paleta alinhada à do notebook de análise (padrão do artigo):
    #   R-NSGA-II azul | PI-NSGA-II vermelho
    dados = [("R-NSGA-II",  fase2[nome], "#0072B2"),
             ("PI-NSGA-II", fase3[nome], "#A81313")]

    fig = plt.figure(figsize=(11, 4.6))
    for i, (alg, fase, cor) in enumerate(dados, 1):
        ax = fig.add_subplot(1, 2, i, projection="3d" if n_obj >= 3 else None)
        Fs = [F for F in fase["frentes"] if F is not None]
        Fn = normalizar(np.vstack(Fs), ideal_g, nadir_g)
        Ks = normalizar(np.vstack([k for k in fase["knees"] if k is not None]), ideal_g, nadir_g)
        Kr = normalizar(fase1[nome]["K_ref"], ideal_g, nadir_g).ravel()

        kw_ref  = dict(s=8,  c="#BBBBBB", zorder=1)
        kw_frnt = dict(s=8,  c=cor, alpha=.25, zorder=2)
        kw_kns  = dict(s=55, c=cor, marker="^", edgecolors="white", linewidths=.6, zorder=4)
        kw_kref = dict(s=120, c="k", marker="X", zorder=5)
        kw_idl  = dict(s=250, c="#E0A300", marker="*", edgecolors="k", linewidths=.5, zorder=6)

        if n_obj >= 3:
            ax.scatter(P_ref[:,0], P_ref[:,1], P_ref[:,2], **kw_ref,  label="Fronteira de referência (NSGA-II)")
            ax.scatter(Fn[:,0],   Fn[:,1],   Fn[:,2],   **kw_frnt, label=f"Soluções do {alg}")
            ax.scatter(Ks[:,0],   Ks[:,1],   Ks[:,2],   **kw_kns,  label="Joelhos por execução (21)")
            ax.scatter(*Kr[:3],                          **kw_kref, label="Joelho de referência")
            ax.scatter(0, 0, 0,                          **kw_idl,  label="Ponto ideal")
        else:
            ax.scatter(P_ref[:,0], P_ref[:,1], **kw_ref,  label="Fronteira de referência (NSGA-II)")
            ax.scatter(Fn[:,0],   Fn[:,1],   **kw_frnt, label=f"Soluções do {alg}")
            ax.scatter(Ks[:,0],   Ks[:,1],   **kw_kns,  label="Joelhos por execução (21)")
            ax.scatter(Kr[0],     Kr[1],     **kw_kref, label="Joelho de referência")
            ax.scatter(0,         0,         **kw_idl,  label="Ponto ideal")
            ax.set_xlim(-.05, 1.05); ax.set_ylim(-.05, 1.05)

        ax.set_title(f"{alg} — {nome}"); ax.legend(fontsize=7, loc="best")

    fig.tight_layout(); plt.show()

for nome in PROBLEMS:
    plot_problema(nome)

In [ ]:
# ============ Exploração interativa 3D — apenas Carside ============
try:
    import plotly.graph_objects as go

    def explorar_3d_carside():
        nome = "Carside"
        m = metricas[nome]
        ideal_g, nadir_g = m["ideal_g"], m["nadir_g"]

        P_ref = normalizar(fase1[nome]["P_ref"], ideal_g, nadir_g)
        Kr    = normalizar(fase1[nome]["K_ref"],  ideal_g, nadir_g).ravel()

        fig = go.Figure()

        # Fundo: fronteira de referência do NSGA-II
        fig.add_trace(go.Scatter3d(
            x=P_ref[:,0], y=P_ref[:,1], z=P_ref[:,2], mode="markers",
            marker=dict(size=2.5, color="#BBBBBB", opacity=0.5),
            name="P_ref (NSGA-II)"))

        # Nuvem + knees de cada algoritmo de preferência
        for alg, fase, cor in [("R-NSGA-II",  fase2[nome], "#0072B2"),
                                ("PI-NSGA-II", fase3[nome], "#A81313")]:
            Fs = [F for F in fase["frentes"] if F is not None]
            Fn = normalizar(np.vstack(Fs), ideal_g, nadir_g)
            Ks = normalizar(np.vstack([k for k in fase["knees"] if k is not None]),
                            ideal_g, nadir_g)

            fig.add_trace(go.Scatter3d(
                x=Fn[:,0], y=Fn[:,1], z=Fn[:,2], mode="markers",
                marker=dict(size=2.5, color=cor, opacity=0.2),
                name=f"{alg} (nuvem)"))

            fig.add_trace(go.Scatter3d(
                x=Ks[:,0], y=Ks[:,1], z=Ks[:,2], mode="markers",
                marker=dict(size=5, color=cor, symbol="diamond",
                            line=dict(color="white", width=1)),
                name=f"{alg} — knees (21)"))

        # K_ref — X preto
        fig.add_trace(go.Scatter3d(
            x=[Kr[0]], y=[Kr[1]], z=[Kr[2]], mode="markers",
            marker=dict(size=7, color="black", symbol="x"),
            name="K_ref (NSGA-II)"))

        # Ponto ideal — estrela âmbar
        fig.add_trace(go.Scatter3d(
            x=[0], y=[0], z=[0], mode="markers",
            marker=dict(size=8, color="#E0A300", symbol="diamond",
                        line=dict(color="black", width=1)),
            name="ideal"))

        fig.update_layout(
            title=dict(text="Carside — Frentes 3D (normalizado)", font=dict(size=14)),
            width=820, height=620,
            scene=dict(xaxis_title="f1 (norm.)",
                       yaxis_title="f2 (norm.)",
                       zaxis_title="f3 (norm.)"),
            legend=dict(font=dict(size=11)))
        fig.show()

    explorar_3d_carside()

except ImportError:
    print("plotly não instalado — pip install plotly")

## 9. (Opcional) Retomar de CSVs sem recomputar

In [ ]:
# ============ (Opcional) Retomar de CSVs sem recomputar ============
# Se o kernel caiu depois de alguma fase, rode o Setup + Funções auxiliares e esta célula.
# Ela reconstrói fase1/fase2/fase3 a partir dos CSVs existentes (knees recalculados, barato).
import pandas as pd

def _carregar_fase(caminho, algoritmo):
    if not os.path.exists(caminho): return None
    df = pd.read_csv(caminho)
    out = {}
    for nome in PROBLEMS:
        sub = df[(df["problema"] == nome) & (df["algoritmo"] == algoritmo)]
        if sub.empty: continue
        d = 3 if sub["f3"].notna().any() else 2
        cols = ["f1", "f2", "f3"][:d]
        frentes = [g[cols].to_numpy(float) for _, g in sub.groupby("run")]
        out[nome] = {"frentes": frentes, "knees": [knee_point(F) for F in frentes]}
    return out

_f1 = _carregar_fase(CSV_F1, "NSGA-II")
if _f1 is not None:
    refs = pd.read_csv(CSV_REF)
    fase1 = {}
    for nome, dados in _f1.items():
        P_ref = filtrar_nao_dominados(np.vstack(dados["frentes"]))
        r = refs[refs["problema"] == nome].set_index("tipo")
        d = 3 if not np.isnan(r.loc["K_ref"].get("f3", np.nan)) else 2
        cols = ["f1", "f2", "f3"][:d]
        fase1[nome] = {"frentes": dados["frentes"], "P_ref": P_ref,
                       "K_ref": r.loc["K_ref", cols].to_numpy(float),
                       "regua": (r.loc["regua_min", cols].to_numpy(float),
                                 r.loc["regua_max", cols].to_numpy(float))}
    print("fase1 recarregada de", CSV_F1)

_f2 = _carregar_fase(CSV_F2, "R-NSGA-II")
if _f2 is not None: fase2 = _f2; print("fase2 recarregada de", CSV_F2)
_f3 = _carregar_fase(CSV_F3, "PI-NSGA-II")
if _f3 is not None: fase3 = _f3; print("fase3 recarregada de", CSV_F3)

## Extensões retiradas do protocolo (ficam para trabalhos futuros)

Por decisão de orientação: sensibilidade ao $\varepsilon$, HV/IGD restritos à ROI e varredura do raio saem do escopo do artigo do ENMC. O protocolo publicado se encerra nas três fases acima + análise estatística das distâncias no notebook `Analise_ENMC_KneeRef.ipynb`.